In [ ]:
from Bio.Align import PairwiseAligner, substitution_matrices
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
import numpy as np
from numpy import random as rd
from matplotlib import pyplot as plt

In [ ]:
from src.estimalign import estimalign
from src.logit_link import logit_partial_scores
from src.optimization import create_powerstep, create_constant_step

# Data

In [ ]:
from miRBench.dataset import list_datasets, get_dataset_df

In [ ]:
hejret_train = get_dataset_df(list_datasets()[0], split="train")
hejret_test = get_dataset_df(list_datasets()[0], split="test")

In [ ]:
mirlist = hejret_train['noncodingRNA']
mirlist = [Seq(seq) for seq in mirlist]
genelist = hejret_train['gene']
genelist = [Seq(seq).reverse_complement() for seq in genelist]

In [ ]:
hejret_train

# Optimization

### Simple model on miRNA alignments:

In [ ]:
true_match = 1
true_mismatch = -1
true_gapopen = -1.2
true_gapext = -0.1

In [ ]:
aligner = PairwiseAligner()
aligner.mode = 'local'
aligner.open_gap_score = true_gapopen
aligner.extend_gap_score = true_gapext
aligner.match = true_match
aligner.mismatch = true_mismatch
# aligner.end_gap_score = 0

In [ ]:
mirlist[0]

In [ ]:
genelist[0]

In [ ]:
print(next(aligner.align(mirlist[1], genelist[1])))

In [ ]:
scores = np.array([aligner.score(a, b) for a, b in zip(mirlist, genelist)])

In [ ]:
plt.figure()
plt.hist(scores, bins=100)
plt.show()

In [ ]:
true_alpha = -9

In [ ]:
logit_scores = logit_partial_scores(scores, true_alpha)

In [ ]:
plt.figure()
plt.hist(logit_scores, bins=100)
plt.show()

In [ ]:
rd.rand(10)

In [ ]:
labels = rd.rand(len(mirlist))
labels = labels <= logit_scores
labels

In [ ]:
true_logL = np.sum(np.log(logit_scores[labels]))+np.sum(np.log(1-logit_scores[~labels]))
print('Sum of log-logit scores:', np.sum(np.log(logit_scores)))
print('True LogL:', true_logL)

In [ ]:
plt.figure()
plt.plot(scores, labels, 'r.', alpha=0.005)

In [ ]:
plt.figure()
plt.plot(logit_scores, labels, 'r.', alpha=0.005)

In [ ]:
const_step = create_constant_step(0.00005)
# powerstep = create_powerstep(0.00001, power=0.5, burnin=0)
# powerstep = create_powerstep(0.00001, power=-0.5, burnin=0)

In [ ]:
NITER = 5 # original 50

In [ ]:
# Task 1: Run estimalign across different step sizes and plot:
# (1) logL trajectories to see convergence behavior
# (2) scatter of convergence speed vs final parameter error to find the best step size

step_sizes = [0.000005, 0.00001, 0.00005, 0.0001, 0.0005]
task1_results = {}

for ss in step_sizes:
    step = create_constant_step(ss)
    p = estimalign(mirlist, genelist, labels,
                   stepfunction=step, aligner_mode='local',
                   substitution_mode='simple', verbose=False,
                   max_iter=NITER, stochastic_factor=None, num_threads=16)
    task1_results[ss] = p

# Trajectory plot
plt.figure()
for ss, p in task1_results.items():
    plt.plot(p['loglik_trajectory'], label=f'step={ss}')
plt.axhline(true_logL, color='k', linestyle='--', label='True LogL')
plt.xlabel('Iteration'); plt.ylabel('LogL')
plt.title('LogL Trajectories by Step Size')
plt.legend(); plt.tight_layout(); plt.show()

# Scatter: convergence speed vs final parameter error
def param_error(p):
    return (abs(p['match_score'] - true_match) +
            abs(p['mismatch_score'] - true_mismatch) +
            abs(p['open_gap_score'] - true_gapopen) +
            abs(p['extend_gap_score'] - true_gapext) +
            abs(p['alpha'] - true_alpha))

# convergence = first iteration where subgradient L2 norm drops below threshold
threshold = 0.01
conv_iters = []
errors = []
for ss, p in task1_results.items():
    grads = p['subgradient_l2_trajectory']
    conv = next((i for i, g in enumerate(grads) if g < threshold), NITER)
    conv_iters.append(conv)
    errors.append(param_error(p))

plt.figure()
for i, ss in enumerate(step_sizes):
    plt.scatter(conv_iters[i], errors[i], label=f'step={ss}', s=100)
plt.xlabel('Iterations to Converge')
plt.ylabel('Total Parameter Error')
plt.title('Convergence Speed vs Parameter Error')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
params = estimalign(mirlist, genelist, labels, 
                    stepfunction=const_step,
                    aligner_mode='local',
                    substitution_mode='simple',
                    verbose=True, max_iter=NITER,
                    stochastic_factor=None,
                    num_threads = 16)

In [ ]:
print(params['final_loglik'])

In [ ]:
print(params['final_loglik'])

In [ ]:
plt.figure()
plt.subplot(221)
plt.plot(np.arange(NITER), params['subgradient_l2_trajectory'])
plt.plot([0, NITER], [0, 0], 'k--')
plt.title('Subgradient L2 norm trajectory')


plt.subplot(222)
plt.plot(np.arange(NITER+1), params['loglik_trajectory'])
plt.plot([0, NITER+1], [true_logL, true_logL], 'k--')
plt.title('LogLikelihood trajectory')

plt.subplot(223)
plt.plot(np.arange(NITER//2, NITER), params['subgradient_l2_trajectory'][NITER//2:])
plt.plot([NITER//2, NITER], [0, 0], 'k--')
plt.title('Subgradient L2 norm trajectory')

plt.subplot(224)
plt.plot(np.arange(NITER//2, NITER+1), params['loglik_trajectory'][NITER//2:])
plt.plot([NITER//2, NITER+1], [true_logL, true_logL], 'k--')
plt.title('LogLikelihood trajectory')

plt.tight_layout()

In [ ]:
print(true_match, params['match_score'])
print(true_mismatch, params['mismatch_score'])
print(true_gapopen, params['open_gap_score'])
print(true_gapext, params['extend_gap_score'])
print(true_alpha, params['alpha'])

In [ ]:
# Task 3: Run estimalign 20 times with different random seeds and plot:
# 1) a histogram of total parameter error per run
# 2) per-parameter boxplots to show which parameters are reliably recovered

N_SEEDS = 20
best_step = 0.000005 #using best step size from task 1
param_names = ['match_score', 'mismatch_score', 'open_gap_score', 'extend_gap_score', 'alpha']
true_values = [true_match, true_mismatch, true_gapopen, true_gapext, true_alpha]

per_param_errors = {name: [] for name in param_names}
total_errors = []

for seed in range(N_SEEDS):
    rd.seed(seed)
    ls = logit_partial_scores(scores, true_alpha)
    lbs = rd.rand(len(mirlist)) <= ls
    p = estimalign(mirlist, genelist, lbs,
                   stepfunction=create_constant_step(best_step),
                   aligner_mode='local', substitution_mode='simple',
                   verbose=False, max_iter=NITER,
                   stochastic_factor=None, num_threads=16)
    
    total = 0
    for name, true_val in zip(param_names, true_values):
        err = abs(p[name] - true_val)
        per_param_errors[name].append(err)
        total += err
    total_errors.append(total)

# Histogram of total error per run
plt.figure()
plt.hist(total_errors, bins=10, edgecolor='black')
plt.xlabel('Total Parameter Error')
plt.ylabel('Count')
plt.title('Distribution of Total Parameter Error Across 20 Runs')
plt.tight_layout(); plt.show()

# Per-parameter boxplots
plt.figure()
plt.boxplot(per_param_errors.values(), labels=param_names, showfliers=True)
plt.ylabel('Absolute Error')
plt.title('Per-Parameter Recovery Error Across 20 Runs')
plt.xticks(rotation=15)
plt.tight_layout(); plt.show()

### General matrix, affine gap penalty

In [ ]:
true_gapopen = -1.2
true_gapext = -0.1
true_substitution = substitution_matrices.Array(alphabet='ACTG', 
                                          data=np.array([
                                              [1, -0.3, -1, -0.8], 
                                              [-0.6, 1.2, -0.3, -1], 
                                              [-1.2, -0.4, 1, -0.8], 
                                              [-0.4, -1.4, -0.9, 1.3]]))

In [ ]:
aligner = PairwiseAligner()
aligner.mode = 'local'
aligner.open_gap_score = true_gapopen
aligner.extend_gap_score = true_gapext
aligner.substitution_matrix = true_substitution

In [ ]:
scores = np.array([aligner.score(a, b) for a, b in zip(mirlist, genelist)])

In [ ]:
plt.figure()
plt.hist(scores, bins=100)
plt.show()

In [ ]:
true_alpha = -12
logit_scores = logit_partial_scores(scores, true_alpha)

In [ ]:
plt.figure()
plt.hist(logit_scores, bins=100)
plt.show()

In [ ]:
labels = rd.rand(len(mirlist))
labels = labels <= logit_scores
true_logL = np.sum(np.log(logit_scores[labels]))+np.sum(np.log(1-logit_scores[~labels]))
print('Sum of log-logit scores:', np.sum(np.log(logit_scores)))
print('True LogL:', true_logL)

In [ ]:
const_step = create_constant_step(0.00005)
# powerstep = create_powerstep(0.00005, power=0.5, burnin=0)
powerstep = create_powerstep(0.00002, power=-0.1, burnin=0)

In [ ]:
NITER = 200

In [ ]:
params = estimalign(mirlist, genelist, labels, 
                    stepfunction=const_step,
                    aligner_mode='local',
                    substitution_mode='general',
                    gap_mode='affine', 
                    stochastic_factor=0.01,
                    verbose=True, max_iter=NITER,
                    num_threads = 24)

In [ ]:
print(params['final_loglik'])

In [ ]:
plt.figure()
plt.subplot(221)
plt.plot(np.arange(NITER), params['subgradient_l2_trajectory'])
plt.plot([0, NITER], [0, 0], 'k--')
plt.title('Subgradient L2 norm trajectory')


plt.subplot(222)
plt.plot(np.arange(NITER+1), params['loglik_trajectory'])
plt.plot([0, NITER+1], [true_logL, true_logL], 'k--')
plt.title('LogLikelihood trajectory')


plt.subplot(223)
plt.plot(np.arange(NITER//2, NITER), params['subgradient_l2_trajectory'][NITER//2:])
plt.plot([NITER//2, NITER], [0, 0], 'k--')
plt.title('Subgradient L2 norm trajectory')

plt.subplot(224)
plt.plot(np.arange(NITER//2, NITER+1), params['loglik_trajectory'][NITER//2:])
plt.plot([NITER//2, NITER+1], [true_logL, true_logL], 'k--')
plt.title('LogLikelihood trajectory')

plt.tight_layout()

In [ ]:
print(true_gapopen, params['open_gap_score'])
print(true_gapext, params['extend_gap_score'])
print(true_alpha, params['alpha'])
true_subs_vector = []
param_subs_vector = []
for char1 in true_substitution.alphabet:
    for char2 in true_substitution.alphabet:
        true_v = true_substitution[char1, char2]
        true_subs_vector.append(true_v)
        estim_v = params['substitution_matrix'][char1, char2]
        param_subs_vector.append(estim_v)
        print(char1, char2, true_v, estim_v)
        
print(np.corrcoef(true_subs_vector, param_subs_vector))
print(np.mean(np.abs(np.array(true_subs_vector)- np.array(param_subs_vector))))

In [ ]:
plt.figure()
plt.plot(true_subs_vector, param_subs_vector, 'r.')

In [ ]:
from src.optimization import get_initial_estimate

In [ ]:
get_initial_estimate(params['alignments'], labels, substitution_mode='general', gap_mode='affine',
                     alphabet=true_substitution.alphabet)

In [ ]:
true_substitution

In [ ]:
estim_substitution = true_substitution.copy()
for char1 in true_substitution.alphabet:
    for char2 in true_substitution.alphabet:
        estim_substitution[char1, char2] = params['substitution_matrix'][char1, char2]

In [ ]:
estim_substitution

# Proteins

### Simulated alignments

In [ ]:
blosum62 = substitution_matrices.load('BLOSUM62')

In [ ]:
coupling = 2**blosum62/(len(blosum62.alphabet)**2)
coupling /= np.sum(coupling)

In [ ]:
aa_pairs = [(char1, char2) for char1 in blosum62.alphabet for char2 in blosum62.alphabet]
prob_vect = [coupling[aapair] for aapair in aa_pairs]

In [ ]:
for aa, prob in zip(aa_pairs, prob_vect):
    print(aa, prob)

In [ ]:
SETSIZE = 5000

In [ ]:
Alist = []
Blist = []
for _ in range(SETSIZE):
    pairs = rd.choice(len(aa_pairs), p=prob_vect, replace=True, size=250)
    pairs = [aa_pairs[i] for i in pairs]
    aseq = ''.join(x[0] for x in pairs)
    bseq = ''.join(x[1] for x in pairs)
    Alist.append(aseq)
    Blist.append(bseq)

In [ ]:
Blist

### Symmetric matrix on protein alignments:

In [ ]:
positive_id_tuples = []
close_negative_id_tuples = []
with open('./Proteomes/human_to_chicken_upto1000aa.blast') as h:
    for l in h:
        l = l.split('\t')
        evalue = float(l[-2])
        if evalue == 0:
            positive_id_tuples.append([l[0], l[1]])
        elif evalue > 0.0001:
            close_negative_id_tuples.append([l[0], l[1]])

In [ ]:
human_proteome = list(SeqIO.parse('Proteomes/GCF_000001405.40/up_to_1000.faa', 'fasta'))
chick_proteome = list(SeqIO.parse('Proteomes/GCF_016699485.2/up_to_1000.faa', 'fasta'))

In [ ]:
alphabet = 'ACDEFGHIKLMNPQRSTVWY'

In [ ]:
human_proteome = [seq for seq in human_proteome if set(str(seq.seq)).issubset(set(alphabet))]
chick_proteome = [seq for seq in chick_proteome if set(str(seq.seq)).issubset(set(alphabet))]

In [ ]:
human_prot_ids = [seq.id for seq in human_proteome]
chick_prot_ids = [seq.id for seq in chick_proteome]

In [ ]:
human_prot_ids_set = set(human_prot_ids)
chick_prot_ids_set = set(chick_prot_ids)

In [ ]:
positive_id_tuples = [t for t in positive_id_tuples if t[0] in human_prot_ids_set and t[1] in chick_prot_ids_set]
# close_negative_id_tuples = [t for t in close_negative_id_tuples if t[0] in human_prot_ids_set and t[1] in chick_prot_ids_set]

In [ ]:
SETSIZE = 5000

In [ ]:
# pos_set = rd.choice(len(positive_id_tuples), SETSIZE//2, replace=False)
# neg_set = rd.choice(len(close_negative_id_tuples), SETSIZE//2, replace=False)
# prot_dset = [positive_id_tuples[i] for i in pos_set] + [close_negative_id_tuples[i] for i in neg_set]
pos_set = rd.choice(len(positive_id_tuples), SETSIZE, replace=False)
prot_dset = [positive_id_tuples[i] for i in pos_set]

In [ ]:
human_list = [human_proteome[human_prot_ids.index(hpid)] for hpid, cpid in prot_dset]
chick_list = [chick_proteome[chick_prot_ids.index(cpid)] for hpid, cpid in prot_dset]

In [ ]:
blosum62 = substitution_matrices.load('BLOSUM62')

In [ ]:
blosum62 /= np.sqrt(np.sum(blosum62**2))

In [ ]:
aligner = PairwiseAligner()
aligner.mode = 'global'
aligner.substitution_matrix=blosum62
aligner.open_gap_score = -1
aligner.extend_gap_score = -0.1

Example scores of alignments to visualize gap open and extend penalties for global alignment:

In [ ]:
test_aln = aligner.align('ATA', 'AA')
print(next(test_aln))
print(2*blosum62['A', 'A'] - 1, test_aln.score)
test_aln = aligner.align('ATTA', 'AA')
print(next(test_aln))
print(2*blosum62['A', 'A'] - 1 - 0.1, test_aln.score)

In [ ]:
scores = np.array([aligner.score(a, b) for a, b in zip(human_list, chick_list)])

In [ ]:
plt.figure()
plt.hist(scores, bins=100)
plt.show()

In [ ]:
print(next(aligner.align(human_list[0], chick_list[0])))

In [ ]:
logit_scores = logit_partial_scores(scores, -20)

In [ ]:
plt.figure()
plt.hist(logit_scores, bins=100)
plt.show()

Expectation check:

In [ ]:
logL_distribution = []
for _ in range(5000):
    labels = rd.rand(len(human_list))
    labels = labels <= logit_scores
    true_logL = np.sum(np.log(logit_scores[labels]))+np.sum(np.log(1-logit_scores[~labels]))
    logL_distribution.append(true_logL)

In [ ]:
plt.figure()
plt.hist(logL_distribution, bins=100)
plt.show()

In [ ]:
EL = 0 
VL = 0
for ls in logit_scores:
    if 1e-24 < ls < 1-1e-24:
        EL += ls*np.log(ls) + (1-ls)*np.log(1-ls)
        VL += ls*(1-ls)*(np.log(ls)**2 + np.log(1-ls)**2) # Incorrect
SDL = np.sqrt(VL)

In [ ]:
print('Expected LogL:', EL)
print('Average LogL:', np.mean(logL_distribution))
print('STD LogL:', SDL)
print('Sample STD:', np.std(logL_distribution))

Fitting:

In [ ]:
labels = rd.rand(len(human_list))
labels = labels <= logit_scores

In [ ]:
true_logL = np.sum(np.log(logit_scores[labels]))+np.sum(np.log(1-logit_scores[~labels]))

In [ ]:
print('Sum of log-logit scores:', np.sum(np.log(logit_scores)))
print('True LogL:', true_logL)

In [ ]:
plt.figure()
plt.plot(logit_scores, labels, 'r.', alpha=0.5)

In [ ]:
const_step = create_constant_step(0.00005)
powerstep = create_powerstep(0.000005, power=0.5, burnin=0) # step 0.00005 good for SETSIZE==1000
# powerstep = create_powerstep(0.0000001, power=-0.5, burnin=0)

In [ ]:
NITER = 50

In [ ]:
set('WEIMFASVCGKLTHDPRNQY') == set(alphabet) 

In [ ]:
params = estimalign(human_list, chick_list, labels, 
                    stepfunction=powerstep,
                    aligner_mode='global',
                    substitution_mode='general',
                    gap_mode='affine',
                    baseline_aligner=aligner,
                    stochastic_factor=0.0001,
                    verbose=True, max_iter=NITER,
                    num_threads = 16)

In [ ]:
print(params['final_loglik'])

In [ ]:
print(max(params['loglik_trajectory']))

In [ ]:
plt.figure()
plt.subplot(221)
plt.plot(np.arange(NITER), params['subgradient_l2_trajectory'])
plt.plot([0, NITER], [0, 0], 'k--')
plt.title('Subgradient L2 norm trajectory')


plt.subplot(222)
plt.plot(np.arange(NITER+1), params['loglik_trajectory'])
plt.plot([0, NITER+1], [true_logL, true_logL], 'k--')
plt.title('LogLikelihood trajectory')

plt.subplot(223)
plt.plot(np.arange(NITER//2, NITER), params['subgradient_l2_trajectory'][NITER//2:])
plt.plot([NITER//2, NITER], [0, 0], 'k--')
plt.title('Subgradient L2 norm trajectory')

plt.subplot(224)
plt.plot(np.arange(NITER//2, NITER+1), params['loglik_trajectory'][NITER//2:])
plt.plot([NITER//2, NITER+1], [true_logL, true_logL], 'k--')
plt.title('LogLikelihood trajectory')

plt.tight_layout()

In [ ]:
print(params['open_gap_score'], aligner.open_gap_score)
print(params['extend_gap_score'], aligner.extend_gap_score)

In [ ]:
params['alpha']

In [ ]:
blosum_vs_mine = []
for char1 in params['substitution_matrix'].alphabet:
    for char2 in params['substitution_matrix'].alphabet:
        if char1 != 'N' and char2 != 'N':
            blosum_vs_mine.append([char1, char2,blosum62[char1, char2], params['substitution_matrix'][char1, char2]])

In [ ]:
print('Blosum, Mine')
for i in rd.choice(len(blosum_vs_mine), 10, replace=False):
    print(blosum_vs_mine[i])

In [ ]:
np.corrcoef([x[2] for x in blosum_vs_mine], [x[3] for x in blosum_vs_mine])

In [ ]:
plt.figure()
plt.plot([x[2] for x in blosum_vs_mine], [x[3] for x in blosum_vs_mine], 'r.')

In [ ]:
plt.figure()
plt.plot(sorted([x[2] for x in blosum_vs_mine]), sorted([x[3] for x in blosum_vs_mine]), 'r.')